In [0]:
from pyspark.sql.functions import col, to_timestamp, to_date, year, month, dayofmonth, dayofweek, hour, when, round
from pyspark.sql.types import DecimalType

print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


In [0]:
# Load data from Bronze tables
print("=== LOADING BRONZE TABLES ===\n")

customers_bronze = spark.table("food_delivery.bronze.customers")
restaurants_bronze = spark.table("food_delivery.bronze.restaurants")
drivers_bronze = spark.table("food_delivery.bronze.drivers")
orders_bronze = spark.table("food_delivery.bronze.orders")
order_items_bronze = spark.table("food_delivery.bronze.order_items")
payments_bronze = spark.table("food_delivery.bronze.payments")

print(f"✅ Customers: {customers_bronze.count():,} rows")
print(f"✅ Restaurants: {restaurants_bronze.count():,} rows")
print(f"✅ Drivers: {drivers_bronze.count():,} rows")
print(f"✅ Orders: {orders_bronze.count():,} rows")
print(f"✅ Order Items: {order_items_bronze.count():,} rows")
print(f"✅ Payments: {payments_bronze.count():,} rows")

=== LOADING BRONZE TABLES ===

✅ Customers: 1,000 rows
✅ Restaurants: 100 rows
✅ Drivers: 200 rows
✅ Orders: 10,000 rows
✅ Order Items: 25,045 rows
✅ Payments: 10,000 rows


In [0]:
print("=== FIXING DATA TYPES ===\n")

# 1. CUSTOMERS - No date columns to fix, just keep as is
customers_silver = customers_bronze
print("✅ Customers: No date columns to fix")

# 2. RESTAURANTS - No date columns to fix, just keep as is
restaurants_silver = restaurants_bronze
print("✅ Restaurants: No date columns to fix")

# 3. DRIVERS - No date columns to fix, just keep as is
drivers_silver = drivers_bronze
print("✅ Drivers: No date columns to fix")

# 4. ORDERS - Fix timestamp and numeric columns
orders_silver = orders_bronze \
    .withColumn("order_time", to_timestamp(col("order_time"), "yyyy-MM-dd HH:mm:ss")) \
    .withColumn("delivery_distance_km", col("delivery_distance_km").cast("double")) \
    .withColumn("discount", col("discount").cast("decimal(10,2)")) \
    .withColumn("tax", col("tax").cast("decimal(10,2)")) \
    .withColumn("total_amount", col("total_amount").cast("decimal(10,2)"))

print("✅ Orders: Fixed timestamp and numeric columns")

# 5. ORDER_ITEMS - Fix numeric columns
order_items_silver = order_items_bronze \
    .withColumn("quantity", col("quantity").cast("int")) \
    .withColumn("unit_price", col("unit_price").cast("decimal(10,2)"))

print("✅ Order Items: Fixed numeric columns")

# 6. PAYMENTS - Fix numeric columns
payments_silver = payments_bronze \
    .withColumn("amount", col("amount").cast("decimal(10,2)"))

print("✅ Payments: Fixed numeric columns")

print("\n✅ All data types fixed successfully!")

=== FIXING DATA TYPES ===

✅ Customers: No date columns to fix
✅ Restaurants: No date columns to fix
✅ Drivers: No date columns to fix
✅ Orders: Fixed timestamp and numeric columns
✅ Order Items: Fixed numeric columns
✅ Payments: Fixed numeric columns

✅ All data types fixed successfully!


In [0]:
print("=== HANDLING NULL VALUES ===\n")

# CUSTOMERS - Fill missing values
customers_silver = customers_silver \
    .fillna({"name": "Unknown", "city": "Unknown", "email": "unknown@email.com"})
print("✅ Customers: NULL values handled")

# RESTAURANTS - Fill missing values
restaurants_silver = restaurants_silver \
    .fillna({"restaurant_name": "Unknown", "city": "Unknown", "cuisine": "Unknown"})
print("✅ Restaurants: NULL values handled")

# DRIVERS - Fill missing values
drivers_silver = drivers_silver \
    .fillna({"driver_name": "Unknown", "vehicle": "Unknown", "availability": "Unknown"})
print("✅ Drivers: NULL values handled")

# ORDERS - Handle NULLs
orders_silver = orders_silver \
    .fillna({"order_status": "pending"}) \
    .fillna({"delivery_distance_km": 0.0, "discount": 0.0, "tax": 0.0, "total_amount": 0.0}) \
    .fillna({"traffic_level": "Unknown"})
print("✅ Orders: NULL values handled")

# ORDER_ITEMS - No NULLs to fill (these are critical fields)
print("✅ Order Items: No NULLs to handle")

# PAYMENTS - Fill missing values
payments_silver = payments_silver \
    .fillna({"payment_method": "Unknown", "payment_status": "pending"}) \
    .fillna({"amount": 0.0})
print("✅ Payments: NULL values handled")

print("\n✅ All NULL values handled successfully!")

=== HANDLING NULL VALUES ===

✅ Customers: NULL values handled
✅ Restaurants: NULL values handled
✅ Drivers: NULL values handled
✅ Orders: NULL values handled
✅ Order Items: No NULLs to handle
✅ Payments: NULL values handled

✅ All NULL values handled successfully!


In [0]:
print("=== ADDING DERIVED COLUMNS ===\n")

# ORDERS - Add date/time dimensions
orders_silver = orders_silver \
    .withColumn("order_date", to_date(col("order_time"))) \
    .withColumn("order_year", year(col("order_time"))) \
    .withColumn("order_month", month(col("order_time"))) \
    .withColumn("order_day", dayofmonth(col("order_time"))) \
    .withColumn("order_dayofweek", dayofweek(col("order_time"))) \
    .withColumn("order_hour", hour(col("order_time"))) \
    .withColumn("is_weekend", when(dayofweek(col("order_time")).isin([1, 7]), 1).otherwise(0)) \
    .withColumn("net_total", col("total_amount") - col("discount") + col("tax"))

print("✅ Orders: Added date/time dimensions and net_total")

# PAYMENTS - No derived columns needed
print("✅ Payments: No derived columns needed")

# ORDER_ITEMS - Add total price calculation
order_items_silver = order_items_silver \
    .withColumn("total_price", col("quantity") * col("unit_price"))

print("✅ Order Items: Added total_price calculation")

print("\n✅ All derived columns added successfully!")

=== ADDING DERIVED COLUMNS ===

✅ Orders: Added date/time dimensions and net_total
✅ Payments: No derived columns needed
✅ Order Items: Added total_price calculation

✅ All derived columns added successfully!


In [0]:
print("=== SILVER DATA VALIDATION ===\n")

def validate_silver_table(df, table_name):
    total_rows = df.count()
    null_count = 0
    null_columns = []
    
    for col_name in df.columns:
        nulls = df.filter(col(col_name).isNull()).count()
        if nulls > 0:
            null_count += nulls
            null_columns.append(f"{col_name} ({nulls})")
    
    print(f"\n📊 {table_name.upper()}:")
    print(f"   Total rows: {total_rows:,}")
    print(f"   Total columns: {len(df.columns)}")
    
    if null_count == 0:
        print(f"   ✅ NULL values: None found!")
    else:
        print(f"   ⚠️ NULL values: {null_count} found in: {', '.join(null_columns[:3])}")
        if len(null_columns) > 3:
            print(f"      ... and {len(null_columns) - 3} more")
    
    print(f"   📝 Sample data:")
    display(df.limit(3))
    print("-" * 60)

# Validate all Silver DataFrames
validate_silver_table(customers_silver, "customers")
validate_silver_table(restaurants_silver, "restaurants")
validate_silver_table(drivers_silver, "drivers")
validate_silver_table(orders_silver, "orders")
validate_silver_table(order_items_silver, "order_items")
validate_silver_table(payments_silver, "payments")

=== SILVER DATA VALIDATION ===


📊 CUSTOMERS:
   Total rows: 1,000
   Total columns: 4
   ✅ NULL values: None found!
   📝 Sample data:


customer_id,name,city,email
11,Customer 11,Ajman,customer11@mail.com
25,Customer 25,Ajman,customer25@mail.com
103,Customer 103,Al Ain,customer103@mail.com


------------------------------------------------------------

📊 RESTAURANTS:
   Total rows: 100
   Total columns: 4
   ✅ NULL values: None found!
   📝 Sample data:


restaurant_id,restaurant_name,city,cuisine
42,Restaurant 42,Dubai,Italian
45,Restaurant 45,Abu Dhabi,Burger
86,Restaurant 86,Ajman,Arabic


------------------------------------------------------------

📊 DRIVERS:
   Total rows: 200
   Total columns: 4
   ✅ NULL values: None found!
   📝 Sample data:


...

------------------------------------------------------------

📊 ORDER_ITEMS:
   Total rows: 25,045
   Total columns: 5
   ✅ NULL values: None found!
   📝 Sample data:


order_id,item_name,quantity,unit_price,total_price
8,Burger,2,37.00,74.00
24,Cola,3,36.00,108.00
29,Biryani,2,37.00,74.00


------------------------------------------------------------

📊 PAYMENTS:
   Total rows: 10,000
   Total columns: 4
   ✅ NULL values: None found!
   📝 Sample data:


order_id,payment_method,amount,payment_status
6,Cash,113.40,Paid
20,Apple Pay,127.30,Paid
46,Apple Pay,101.55,Refunded


------------------------------------------------------------


In [0]:
print("=== SAVING TO SILVER LAYER ===\n")

# Save to Delta tables (Silver layer)
customers_silver.write.format("delta").mode("overwrite").saveAsTable("food_delivery.silver.customers")
restaurants_silver.write.format("delta").mode("overwrite").saveAsTable("food_delivery.silver.restaurants")
drivers_silver.write.format("delta").mode("overwrite").saveAsTable("food_delivery.silver.drivers")
orders_silver.write.format("delta").mode("overwrite").saveAsTable("food_delivery.silver.orders")
order_items_silver.write.format("delta").mode("overwrite").saveAsTable("food_delivery.silver.order_items")
payments_silver.write.format("delta").mode("overwrite").saveAsTable("food_delivery.silver.payments")

print("✅ All Silver tables saved successfully!")
print("📊 Tables created in: food_delivery.silver.*")
print("\n📋 Silver tables:")
print(f"   • food_delivery.silver.customers")
print(f"   • food_delivery.silver.restaurants")
print(f"   • food_delivery.silver.drivers")
print(f"   • food_delivery.silver.orders")
print(f"   • food_delivery.silver.order_items")
print(f"   • food_delivery.silver.payments")

=== SAVING TO SILVER LAYER ===

✅ All Silver tables saved successfully!
📊 Tables created in: food_delivery.silver.*

📋 Silver tables:
   • food_delivery.silver.customers
   • food_delivery.silver.restaurants
   • food_delivery.silver.drivers
   • food_delivery.silver.orders
   • food_delivery.silver.order_items
   • food_delivery.silver.payments


In [0]:
print("=== SILVER TABLE VERIFICATION ===\n")

# List all Silver tables
print("📋 Listing all Silver tables:")
display(spark.sql("SHOW TABLES IN food_delivery.silver"))

# Show row counts
print("\n📊 Silver table row counts:")
silver_counts = spark.sql("""
    SELECT 
        'customers' as table_name, 
        COUNT(*) as row_count 
    FROM food_delivery.silver.customers
    UNION ALL
    SELECT 'restaurants', COUNT(*) 
    FROM food_delivery.silver.restaurants
    UNION ALL
    SELECT 'drivers', COUNT(*) 
    FROM food_delivery.silver.drivers
    UNION ALL
    SELECT 'orders', COUNT(*) 
    FROM food_delivery.silver.orders
    UNION ALL
    SELECT 'order_items', COUNT(*) 
    FROM food_delivery.silver.order_items
    UNION ALL
    SELECT 'payments', COUNT(*) 
    FROM food_delivery.silver.payments
""")
display(silver_counts)

# Preview orders with new columns
print("\n📝 Sample orders with derived columns:")
display(spark.sql("""
    SELECT order_id, order_time, order_date, order_year, order_month, 
           order_dayofweek, order_hour, is_weekend, net_total
    FROM food_delivery.silver.orders 
    LIMIT 5
"""))

=== SILVER TABLE VERIFICATION ===

📋 Listing all Silver tables:


database,tableName,isTemporary
silver,customers,false
silver,drivers,false
silver,order_items,false
silver,orders,false
silver,payments,false
silver,restaurants,false



📊 Silver table row counts:


table_name,row_count
customers,1000
restaurants,100
drivers,200
orders,10000
order_items,25045
payments,10000



📝 Sample orders with derived columns:


order_id,order_time,order_date,order_year,order_month,order_dayofweek,order_hour,is_weekend,net_total
6,2025-06-25T13:58:00.000Z,2025-06-25,2025,6,4,13,0,118.80
11,2025-11-16T01:37:00.000Z,2025-11-16,2025,11,1,1,1,41.60
45,2025-10-10T20:06:00.000Z,2025-10-10,2025,10,6,20,0,196.80
51,2025-11-10T12:48:00.000Z,2025-11-10,2025,11,2,12,0,32.80
108,2025-01-01T22:23:00.000Z,2025-01-01,2025,1,4,22,0,383.80


In [0]:
print("=" * 60)
print("✅ SILVER LAYER COMPLETE!")
print("=" * 60)

print("\n📊 Data Summary:")
print("-" * 40)
print(f"Customers:     {customers_silver.count():>6,} rows, {len(customers_silver.columns)} columns")
print(f"Restaurants:   {restaurants_silver.count():>6,} rows, {len(restaurants_silver.columns)} columns")
print(f"Drivers:       {drivers_silver.count():>6,} rows, {len(drivers_silver.columns)} columns")
print(f"Orders:        {orders_silver.count():>6,} rows, {len(orders_silver.columns)} columns")
print(f"Order Items:   {order_items_silver.count():>6,} rows, {len(order_items_silver.columns)} columns")
print(f"Payments:      {payments_silver.count():>6,} rows, {len(payments_silver.columns)} columns")

print("\n🎯 Silver Layer Transformations Applied:")
print("-" * 40)
print("✅ Fixed data types (timestamps, decimals, integers)")
print("✅ Handled NULL values with appropriate defaults")
print("✅ Added derived columns (date dimensions, calculations)")
print("✅ Created total_price in order_items")
print("✅ Created net_total in orders (total - discount + tax)")
print("✅ Validated data quality")
print("✅ Saved as Delta tables")

print("\n📁 Silver Tables Location:")
print("-" * 40)
print("food_delivery.silver.customers")
print("food_delivery.silver.restaurants")
print("food_delivery.silver.drivers")
print("food_delivery.silver.orders")
print("food_delivery.silver.order_items")
print("food_delivery.silver.payments")

print("\n🚀 Next: Gold Layer - Business Metrics & Analytics")
print("-" * 40)
print("We'll create:")
print("• Revenue by city and cuisine")
print("• Order volume trends")
print("• Customer behavior analysis")
print("• Driver performance metrics")
print("• Time-based analytics")

print("\n" + "=" * 60)
print("✅ Ready for Lesson 3 - Gold Layer!")
print("=" * 60)

✅ SILVER LAYER COMPLETE!

📊 Data Summary:
----------------------------------------
Customers:      1,000 rows, 4 columns
Restaurants:      100 rows, 4 columns
Drivers:          200 rows, 4 columns
Orders:        10,000 rows, 19 columns
Order Items:   25,045 rows, 5 columns
Payments:      10,000 rows, 4 columns

🎯 Silver Layer Transformations Applied:
----------------------------------------
✅ Fixed data types (timestamps, decimals, integers)
✅ Handled NULL values with appropriate defaults
✅ Added derived columns (date dimensions, calculations)
✅ Created total_price in order_items
✅ Created net_total in orders (total - discount + tax)
✅ Validated data quality
✅ Saved as Delta tables

📁 Silver Tables Location:
----------------------------------------
food_delivery.silver.customers
food_delivery.silver.restaurants
food_delivery.silver.drivers
food_delivery.silver.orders
food_delivery.silver.order_items
food_delivery.silver.payments

🚀 Next: Gold Layer - Business Metrics & Analytics
------